# Autonomous Expense Pipeline - Compute-then-Communicate

Run every cell in order (Shift+Enter).

1. **COMPUTE (pandas + scikit-learn)** - cleaning, aggregations, rankings,
   trends, budget vs actual variance, KMeans clustering. **No raw rows go to the LLM.**
2. **COMMUNICATE** - aggregates-only summary -> local `llama3.1:latest` via Ollama.
3. **ARTIFACTS** - charts + narrative under `expense_results/`.
4. **EXCEL REPORT** - clean data + dim/fact tabs under `expense_reports/`.

## Prerequisites
`pandas`, `numpy`, `scikit-learn`, `matplotlib`, `openpyxl`. For LLM:
`ollama serve` and `ollama pull llama3.1:latest`.


---


## 0. Setup
Imports, loads config from `.env`, creates output folders.


In [ ]:
import os, json, textwrap, urllib.request, re
import pymysql
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

ENV_PATH = r"C:\Python\AI-DataSuite\.env"
if os.path.exists(ENV_PATH):
    with open(ENV_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k not in os.environ:
                os.environ[k] = v

EXPENSE_BASE_DIR = os.getenv("EXPENSE_BASE_DIR", r"C:/Python/AI-DataSuite/Data-to-Narrative-Pipelines")
EXPENSE_RESULTS_DIR = os.getenv("EXPENSE_RESULTS_DIR", os.path.join(EXPENSE_BASE_DIR, "expense_results"))
EXPENSE_REPORTS_DIR = os.getenv("EXPENSE_REPORTS_DIR", os.path.join(EXPENSE_BASE_DIR, "expense_reports"))
EXPENSE_OLLAMA_URL = os.getenv("EXPENSE_OLLAMA_URL", "http://localhost:11434/api/chat")
EXPENSE_OLLAMA_MODEL = os.getenv("EXPENSE_OLLAMA_MODEL", "llama3.1:latest")
EXPENSE_NARRATIVE_FILE = os.getenv("EXPENSE_NARRATIVE_FILE", "expense_narrative.txt")
EXPENSE_EXCEL_FILE = os.getenv("EXPENSE_EXCEL_FILE", "Expense_Pipeline_Report.xlsx")
EXPENSE_FACT_CSV = os.getenv("EXPENSE_FACT_CSV", "expense_fact.csv")
EXPENSE_OLLAMA_TIMEOUT = int(os.getenv("EXPENSE_OLLAMA_TIMEOUT", "300"))

# --- MySQL Database Config ---
EXPENSE_DB_HOST = os.getenv("EXPENSE_DB_HOST", "127.0.0.1")
EXPENSE_DB_PORT = int(os.getenv("EXPENSE_DB_PORT", "3306"))
EXPENSE_DB_USER = os.getenv("EXPENSE_DB_USER", "root")
EXPENSE_DB_PASS = os.getenv("EXPENSE_DB_PASS", "")
EXPENSE_DB_NAME = os.getenv("EXPENSE_DB_NAME", "pipeline")
EXPENSE_DB_TABLE = os.getenv("EXPENSE_DB_TABLE", "Expense")

os.makedirs(EXPENSE_RESULTS_DIR, exist_ok=True)
os.makedirs(EXPENSE_REPORTS_DIR, exist_ok=True)
print("Setup complete.")
print("  Results:", EXPENSE_RESULTS_DIR)
print("  Reports:", EXPENSE_REPORTS_DIR)
print("  Ollama:", EXPENSE_OLLAMA_URL, EXPENSE_OLLAMA_MODEL)
print("  MySQL:", EXPENSE_DB_HOST, EXPENSE_DB_PORT, EXPENSE_DB_NAME, EXPENSE_DB_TABLE)


## 1. Load and Clean Data
Loads data from MySQL pipeline.Expense table (already clean DECIMAL/DATE types).


In [ ]:
# Load data from MySQL database
conn = pymysql.connect(
    host=EXPENSE_DB_HOST,
    port=EXPENSE_DB_PORT,
    user=EXPENSE_DB_USER,
    password=EXPENSE_DB_PASS,
    database=EXPENSE_DB_NAME,
    charset='utf8mb4',
)
query = f"SELECT * FROM `{EXPENSE_DB_TABLE}`"
df = pd.read_sql(query, conn)
conn.close()

# Data is already clean from MySQL (DECIMAL/DATE types), no currency cleaning needed
df["Month Date"] = pd.to_datetime(df["Reporting Month"], format="%b-%Y")
df["Month Sort"] = df["Month Date"].dt.to_period("M")
df["Project or Initiative"] = df["Project or Initiative"].fillna("Unassigned")
df["Budget Variance"] = df["Year-to-Date Actual"] - df["Approved Budget"]
df["Budget Variance %"] = np.where(
    df["Approved Budget"] != 0,
    (df["Budget Variance"] / df["Approved Budget"]) * 100,
    0,
)
df["Total Expense"] = (
    df["Salary Expense"] + df["Consulting Expense"] + df["Capital Expense"]
)
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns from MySQL")
print(f"Date range: {df['Reporting Month'].min()} -> {df['Reporting Month'].max()}")
print(f"Business Units: {df['Business Unit'].nunique()}")


## 2. Aggregations and Rankings
Aggregates spend by Business Unit, Category, Cost Centre, Funding Type.


In [ ]:
bu_agg = df.groupby("Business Unit").agg(Approved_Budget=("Approved Budget","sum"),YTD_Actual=("Year-to-Date Actual","sum"),Full_Year_Forecast=("Full-Year Forecast","sum"),Total_Expense=("Total Expense","sum")).reset_index().sort_values("Total_Expense",ascending=False)
bu_agg["Budget Variance"] = bu_agg["YTD_Actual"] - bu_agg["Approved_Budget"]
bu_agg["Budget Variance %"] = np.where(bu_agg["Approved_Budget"]!=0,(bu_agg["Budget Variance"]/bu_agg["Approved_Budget"])*100,0)
cat_agg = df.groupby("Expense Category").agg(YTD_Actual=("Year-to-Date Actual","sum"),Total_Expense=("Total Expense","sum")).reset_index().sort_values("Total_Expense",ascending=False)
cc_agg = df.groupby(["Cost Centre","Cost Centre Description","Business Unit"]).agg(Approved_Budget=("Approved Budget","sum"),YTD_Actual=("Year-to-Date Actual","sum"),Total_Expense=("Total Expense","sum")).reset_index().sort_values("Total_Expense",ascending=False)
cc_agg["Budget Variance"] = cc_agg["YTD_Actual"] - cc_agg["Approved_Budget"]
cc_agg["Budget Variance %"] = np.where(cc_agg["Approved_Budget"]!=0,(cc_agg["Budget Variance"]/cc_agg["Approved_Budget"])*100,0)
fund_agg = df.groupby("Funding Type").agg(Approved_Budget=("Approved Budget","sum"),YTD_Actual=("Year-to-Date Actual","sum"),Total_Expense=("Total Expense","sum")).reset_index().sort_values("Total_Expense",ascending=False)
top10_cc = cc_agg.head(10)
print("Business Unit Aggregation:"); print(bu_agg.to_string(index=False))
print("\nExpense Category Aggregation:"); print(cat_agg.to_string(index=False))
print("\nTop 10 Cost Centres:"); print(top10_cc.to_string(index=False))


## 3. Trend Analysis
Month-over-month spending trends.


In [ ]:
monthly = df.groupby(["Month Sort","Reporting Month"]).agg(Approved_Budget=("Approved Budget","sum"),YTD_Actual=("Year-to-Date Actual","sum"),Total_Expense=("Total Expense","sum")).reset_index().sort_values("Month Sort")
monthly["MoM Change %"] = monthly["Total_Expense"].pct_change()*100
print("Monthly Trend:"); print(monthly.to_string(index=False))


## 4. Budget vs Actual Variance
Compares YTD Actual vs Approved Budget by Business Unit and Funding Type.


In [ ]:
bu_var = bu_agg.copy()
bu_var["Status"] = np.where(bu_var["Budget Variance %"]>5,"OVERSPENT",np.where(bu_var["Budget Variance %"]<-5,"UNDERSPENT","ON TRACK"))
fund_var = fund_agg.copy()
fund_var["Budget Variance"] = fund_var["YTD_Actual"] - fund_var["Approved_Budget"]
fund_var["Budget Variance %"] = np.where(fund_var["Approved_Budget"]!=0,(fund_var["Budget Variance"]/fund_var["Approved_Budget"])*100,0)
fund_var["Status"] = np.where(fund_var["Budget Variance %"]>5,"OVERSPENT",np.where(fund_var["Budget Variance %"]<-5,"UNDERSPENT","ON TRACK"))
overspent_cc = cc_agg[cc_agg["Budget Variance %"]>5].head(10)
underspent_cc = cc_agg[cc_agg["Budget Variance %"]<-5].head(10)
print("Budget Variance by Business Unit:")
print(bu_var[["Business Unit","Approved_Budget","YTD_Actual","Budget Variance","Budget Variance %","Status"]].to_string(index=False))
print("\nBudget Variance by Funding Type:")
print(fund_var[["Funding Type","Approved_Budget","YTD_Actual","Budget Variance","Budget Variance %","Status"]].to_string(index=False))
print("\nTop Overspent Cost Centres:"); print(overspent_cc.to_string(index=False))
print("\nTop Underspent Cost Centres:"); print(underspent_cc.to_string(index=False))


## 5. KMeans Clustering of Cost Centres
Clusters cost centres based on expense features. Uses silhouette score to select k.


In [ ]:
# Add missing expense type columns to cc_agg for clustering
expense_types = df.groupby("Cost Centre").agg(
    Salary_Expense=("Salary Expense","sum"),
    Consulting_Expense=("Consulting Expense","sum"),
    Capital_Expense=("Capital Expense","sum")
).reset_index()
cc_agg = cc_agg.merge(expense_types, on="Cost Centre", how="left")

cluster_features = ["Salary_Expense","Consulting_Expense","Capital_Expense","Total_Expense","Budget Variance"]
cc_model_df = cc_agg[cluster_features].copy().fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cc_model_df)

best_k = 2
best_score = -1
for k in range(2, min(10, len(cc_model_df))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    if score > best_score:
        best_score = score
        best_k = k

print(f"Best k = {best_k} (silhouette score = {best_score:.3f})")

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
cc_agg["Cluster"] = cluster_labels
cc_model_df["Cluster"] = cluster_labels

profile = cc_model_df.groupby("Cluster").mean()
profile["Count"] = cc_model_df.groupby("Cluster").size()

print("\nCluster Profiles:")
print(profile.round(3).to_string())

print("\nCluster Membership:")
for cl in sorted(cc_agg["Cluster"].unique()):
    members = cc_agg.loc[cc_agg["Cluster"] == cl, ["Cost Centre","Cost Centre Description","Business Unit","Total_Expense"]]
    print(f"\nCluster {cl} ({len(members)} cost centres):")
    print(members.sort_values("Total_Expense", ascending=False).to_string(index=False))


## 6. Cluster Visualization (PCA)
Reduces scaled features to 2D via PCA and saves a cluster scatter plot.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
cc_agg["PCA1"] = X_pca[:, 0]
cc_agg["PCA2"] = X_pca[:, 1]
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(cc_agg["PCA1"], cc_agg["PCA2"], c=cc_agg["Cluster"], cmap="viridis", s=100, alpha=0.7, edgecolors="k")
for i, row in cc_agg.iterrows():
    ax.annotate(row["Cost Centre"], (row["PCA1"], row["PCA2"]), fontsize=8, alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Cost Centre Clusters (PCA)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
cluster_plot_path = os.path.join(EXPENSE_RESULTS_DIR, "chart_expense_clusters.png")
plt.savefig(cluster_plot_path, dpi=150)
print(f"Cluster plot saved to: {cluster_plot_path}")

fig2, ax2 = plt.subplots(figsize=(10, 6))
ax2.plot(monthly["Reporting Month"], monthly["Total_Expense"], marker="o", linewidth=2, label="Total Expense")
ax2.plot(monthly["Reporting Month"], monthly["Approved_Budget"], marker="s", linewidth=2, label="Approved Budget", linestyle="--")
ax2.set_xlabel("Reporting Month"); ax2.set_ylabel("Amount")
ax2.set_title("Monthly Expense Trend vs Budget"); ax2.legend()
plt.xticks(rotation=45); plt.tight_layout()
trend_plot_path = os.path.join(EXPENSE_RESULTS_DIR, "chart_monthly_trend.png")
plt.savefig(trend_plot_path, dpi=150)
print(f"Trend plot saved to: {trend_plot_path}")

fig3, ax3 = plt.subplots(figsize=(10, 6))
top_cats_sorted = cat_agg.sort_values("Total_Expense", ascending=True)
ax3.barh(top_cats_sorted["Expense Category"], top_cats_sorted["Total_Expense"], color="#4c72b0")
ax3.set_xlabel("Total Expense"); ax3.set_title("Total Expense by Category")
plt.tight_layout()
cat_plot_path = os.path.join(EXPENSE_RESULTS_DIR, "chart_expense_by_category.png")
plt.savefig(cat_plot_path, dpi=150)
print(f"Category plot saved to: {cat_plot_path}")
plt.show()


## 7. Structured Text Summary
Builds a structured narrative summary of all insights — aggregates only, ready for the LLM.


In [ ]:
lines = []
lines.append("AUTONOMOUS EXPENSE PIPELINE - STRUCTURED SUMMARY")
lines.append("=" * 70)
lines.append("")
lines.append(f"Data source: MySQL pipeline.Expense table ({len(df):,} expense records)")
lines.append(f"Reporting period: {df['Reporting Month'].min()} to {df['Reporting Month'].max()}")
lines.append(f"Fiscal Year: {df['Fiscal Year'].iloc[0]}")
lines.append(f"Business Units: {df['Business Unit'].nunique()} | Cost Centres: {df['Cost Centre'].nunique()}")
lines.append(f"Expense Categories: {df['Expense Category'].nunique()} | Funding Types: {df['Funding Type'].nunique()}")
lines.append("")
lines.append("1. OVERALL SPEND SUMMARY")
total_budget = df["Approved Budget"].sum()
total_actual = df["Year-to-Date Actual"].sum()
total_forecast = df["Full-Year Forecast"].sum()
total_expense = df["Total Expense"].sum()
overall_variance = total_actual - total_budget
overall_variance_pct = (overall_variance / total_budget * 100) if total_budget != 0 else 0
lines.append(f"  Approved Budget:   ${total_budget:,.0f}")
lines.append(f"  YTD Actual:        ${total_actual:,.0f}")
lines.append(f"  Full-Year Forecast: ${total_forecast:,.0f}")
lines.append(f"  Total Expense:     ${total_expense:,.0f}")
lines.append(f"  Budget Variance:   ${overall_variance:,.0f} ({overall_variance_pct:+.1f}%)")
lines.append("")
lines.append("2. SPEND BY BUSINESS UNIT")
for _, r in bu_agg.iterrows():
    lines.append(f"  {r['Business Unit']}: Budget ${r['Approved_Budget']:,.0f} | Actual ${r['YTD_Actual']:,.0f} | Variance ${r['Budget Variance']:,.0f} ({r['Budget Variance %']:+.1f}%)")
lines.append("")
lines.append("3. SPEND BY EXPENSE CATEGORY")
for _, r in cat_agg.iterrows():
    lines.append(f"  {r['Expense Category']}: ${r['Total_Expense']:,.0f}")
lines.append("")
lines.append("4. MONTHLY TREND")
for _, r in monthly.iterrows():
    mom = f"({r['MoM Change %']:+.1f}% MoM)" if pd.notna(r["MoM Change %"]) else "(baseline)"
    lines.append(f"  {r['Reporting Month']}: ${r['Total_Expense']:,.0f} {mom}")
lines.append("")
lines.append("5. BUDGET VARIANCE BY FUNDING TYPE")
for _, r in fund_var.iterrows():
    lines.append(f"  {r['Funding Type']}: Budget ${r['Approved_Budget']:,.0f} | Actual ${r['YTD_Actual']:,.0f} | Variance ${r['Budget Variance']:,.0f} ({r['Budget Variance %']:+.1f}%) [{r['Status']}]")
lines.append("")
lines.append("6. TOP 10 COST CENTRES BY EXPENSE")
for _, r in top10_cc.iterrows():
    lines.append(f"  {r['Cost Centre']} ({r['Business Unit']}): ${r['Total_Expense']:,.0f} | Variance ${r['Budget Variance']:,.0f} ({r['Budget Variance %']:+.1f}%)")
lines.append("")
lines.append("7. CLUSTER PROFILES")
for cl in sorted(cc_agg["Cluster"].unique()):
    members = cc_agg.loc[cc_agg["Cluster"] == cl, ["Cost Centre","Business Unit","Total_Expense"]]
    avg_expense = members["Total_Expense"].mean()
    lines.append(f"  Cluster {cl}: {len(members)} cost centres, avg expense ${avg_expense:,.0f}")
    for _, m in members.iterrows():
        lines.append(f"    - {m['Cost Centre']} ({m['Business Unit']}): ${m['Total_Expense']:,.0f}")
lines.append("")
lines.append("8. KEY INSIGHTS")
top_bu = bu_agg.iloc[0]
lines.append(f"  - Highest spending Business Unit: {top_bu['Business Unit']} (${top_bu['Total_Expense']:,.0f})")
top_cat = cat_agg.iloc[0]
lines.append(f"  - Largest Expense Category: {top_cat['Expense Category']} (${top_cat['Total_Expense']:,.0f})")
if len(overspent_cc) > 0:
    top_overspent = overspent_cc.iloc[0]
    lines.append(f"  - Most overspent Cost Centre: {top_overspent['Cost Centre']} (+{top_overspent['Budget Variance %']:.1f}% over budget)")
else:
    lines.append("  - No cost centres significantly over budget (>5%)")
max_moM = monthly.loc[monthly["MoM Change %"].idxmax()]
lines.append(f"  - Largest MoM increase: {max_moM['Reporting Month']} (+{max_moM['MoM Change %']:.1f}%)")
lines.append("")
summary_text = "\n".join(lines)
print(summary_text)


## 8. COMMUNICATE - Send Summary to Ollama LLM
Sends the structured summary (aggregates only — no raw rows) to the local Ollama model.


In [ ]:
def build_prompt(summary_text: str) -> str:
    return f"""
You are an expert financial analyst writing an expense analysis narrative.
Use ONLY the structured summary below. Do not ask for raw data, and do not
invent institutions, cost centres, or figures.

Write a professional expense analysis report with these sections:
1. Executive Summary
2. Key Insights
3. Spending Trends
4. Budget Variance Analysis
5. Cost Centre Observations
6. Recommendations

Based on this summary:

{summary_text}
""".strip()

prompt = build_prompt(summary_text)

narrative = None
ollama_timeout = EXPENSE_OLLAMA_TIMEOUT
max_retries = 2

for attempt in range(max_retries + 1):
    try:
        payload = {
            "model": EXPENSE_OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0.3},
        }
        req = urllib.request.Request(
            EXPENSE_OLLAMA_URL,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        print(f"Calling Ollama (attempt {attempt + 1}/{max_retries + 1}, timeout={ollama_timeout}s)...")
        with urllib.request.urlopen(req, timeout=ollama_timeout) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        narrative = data["message"]["content"]
        print(f"Narrative received from Ollama ({EXPENSE_OLLAMA_MODEL}).")
        break
    except Exception as e:
        print(f"Ollama attempt {attempt + 1} failed: {e}")
        if attempt < max_retries:
            print("Retrying in 5 seconds...")
            import time
            time.sleep(5)
        else:
            print(f"Ollama not available after {max_retries + 1} attempts. Using local fallback narrative.")
            narrative = textwrap.dedent(f"""
            [LOCAL FALLBACK - Ollama {EXPENSE_OLLAMA_MODEL} was not reachable]
            The compute stage produced the structured summary above. To get the LLM
            narrative, start Ollama, ensure the model '{EXPENSE_OLLAMA_MODEL}' is pulled
            (`ollama pull {EXPENSE_OLLAMA_MODEL}`), then re-run this cell.
            """).strip()

narrative_path = os.path.join(EXPENSE_RESULTS_DIR, EXPENSE_NARRATIVE_FILE)
with open(narrative_path, "w", encoding="utf-8") as f:
    f.write(narrative)

print("\n=== NARRATIVE (first 1500 chars) ===")
print(narrative[:1500])
print("\nFull narrative saved to:", narrative_path)


## 9. EXCEL REPORT - Save Deliverables
Writes clean data, dimension tables, fact table, cluster assignments,
and aggregations to a formatted Excel workbook with charts.


In [ ]:
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference, LineChart

excel_path = os.path.join(EXPENSE_REPORTS_DIR, EXPENSE_EXCEL_FILE)

# --- Ensure Month Sort is string for reliable merging ---
df["Month Sort"] = df["Month Sort"].astype(str)

dim_month = df[["Month Sort","Reporting Month","Fiscal Year"]].drop_duplicates("Month Sort").sort_values("Month Sort").reset_index(drop=True)
dim_month["Month Key"] = range(1, len(dim_month) + 1)
dim_month = dim_month[["Month Key","Month Sort","Reporting Month","Fiscal Year"]]

dim_bu = df[["Business Unit"]].drop_duplicates().reset_index(drop=True)
dim_bu["Business Unit Key"] = range(1, len(dim_bu) + 1)
dim_bu = dim_bu[["Business Unit Key","Business Unit"]]

dim_category = df[["Expense Category"]].drop_duplicates().reset_index(drop=True)
dim_category["Category Key"] = range(1, len(dim_category) + 1)
dim_category = dim_category[["Category Key","Expense Category"]]

dim_funding = df[["Funding Type"]].drop_duplicates().reset_index(drop=True)
dim_funding["Funding Key"] = range(1, len(dim_funding) + 1)
dim_funding = dim_funding[["Funding Key","Funding Type"]]

dim_costcentre = df[["Cost Centre","Cost Centre Description","Business Unit"]].drop_duplicates("Cost Centre").reset_index(drop=True)
dim_costcentre["Cost Centre Key"] = range(1, len(dim_costcentre) + 1)
dim_costcentre = dim_costcentre[["Cost Centre Key","Cost Centre","Cost Centre Description","Business Unit"]]

fact = df.merge(dim_month, on=["Month Sort","Reporting Month","Fiscal Year"])
fact = fact.merge(dim_bu, on="Business Unit")
fact = fact.merge(dim_category, on="Expense Category")
fact = fact.merge(dim_funding, on="Funding Type")
fact = fact.merge(dim_costcentre, on=["Cost Centre","Cost Centre Description","Business Unit"])
fact = fact[[
    "Month Key","Business Unit Key","Category Key","Funding Key","Cost Centre Key",
    "Approved Budget","Year-to-Date Actual","Full-Year Forecast",
    "Prior-Year Actual","Committed Amount","Planned Amount",
    "Position Budget","Actual FTE","Vacant Positions",
    "Salary Expense","Consulting Expense","Capital Expense","Total Expense",
    "Budget Variance","Budget Variance %",
    "Project or Initiative","Data Source","Last Refresh Date",
    ]]
fact.to_csv(os.path.join(EXPENSE_RESULTS_DIR, EXPENSE_FACT_CSV), index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Clean Data", index=False)
    dim_month.to_excel(writer, sheet_name="Dim Month", index=False)
    dim_bu.to_excel(writer, sheet_name="Dim Business Unit", index=False)
    dim_category.to_excel(writer, sheet_name="Dim Category", index=False)
    dim_funding.to_excel(writer, sheet_name="Dim Funding Type", index=False)
    dim_costcentre.to_excel(writer, sheet_name="Dim Cost Centre", index=False)
    fact.to_excel(writer, sheet_name="Fact Expense", index=False)
    bu_agg.to_excel(writer, sheet_name="Agg Business Unit", index=False)
    cat_agg.to_excel(writer, sheet_name="Agg Category", index=False)
    cc_agg.to_excel(writer, sheet_name="Agg Cost Centre", index=False)
    monthly.to_excel(writer, sheet_name="Agg Monthly", index=False)
    fund_var.to_excel(writer, sheet_name="Agg Funding Type", index=False)

from openpyxl import load_workbook
wb = load_workbook(excel_path)
ws = wb["Agg Monthly"]
chart = LineChart()
chart.title = "Monthly Expense Trend vs Budget"
chart.style = 12
chart.y_axis.title = "Amount"
chart.x_axis.title = "Reporting Month"
data = Reference(ws, min_col=4, min_row=1, max_row=len(monthly) + 1)
cats = Reference(ws, min_col=2, min_row=2, max_row=len(monthly) + 1)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
ws.add_chart(chart, "A20")
wb.save(excel_path)

print(f"Excel report saved to: {excel_path}")
print("Sheets: Clean Data, Dim Month, Dim Business Unit, Dim Category,")
print("        Dim Funding Type, Dim Cost Centre, Fact Expense,")
print("        Agg Business Unit, Agg Category, Agg Cost Centre,")
print("        Agg Monthly, Agg Funding Type")


## Pipeline Complete

Artifacts saved:
- `expense_results/` - narrative text + charts
- `expense_reports/` - Excel workbook with dim/fact model

All configuration (paths, file names, Ollama URL/model) is loaded from
`C:/Python/AI-DataSuite/.env` — no credentials are hardcoded in this notebook.
